# Dashboard Migration: Export & Transform (Modular)

## Overview

Consolidated notebook for discovering, exporting, and transforming Lakeview dashboards using centralized configuration and helper modules.

**What this notebook does:**
1. Loads configuration from `config/config.yaml`
2. Discovers dashboards from source workspace
3. Exports dashboard JSONs and permissions
4. Applies catalog/schema transformations using CSV lookup
5. Saves transformed files for manual import

**Next step:** Import dashboards manually (Bundle approach or UI), then run `notebooks/02_Apply_Permissions.ipynb`

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml --quiet
dbutils.library.restartPython()

## Cell 1: Import Helper Modules

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

# Add helpers to path
sys.path.insert(0, '../helpers')

from helpers import (
    load_config,
    get_config,
    get_path,
    create_workspace_client,
    discover_dashboards,
    export_dashboard,
    get_dashboard_permissions,
    load_mapping_csv,
    transform_dashboard_json,
    read_volume_file,
    write_volume_file,
    ensure_directory_exists
)

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
print("="*80)
print("LOADING CONFIGURATION")
print("="*80)

# Load configuration from YAML file
config = load_config('../config/config.yaml')

print("\n✅ Configuration loaded successfully\n")
print(f"   Source workspace: {config['source']['workspace_url']}")
print(f"   Target workspace: {config['target']['workspace_url']}")
print(f"   Volume base: {config['paths']['volume_base']}")
print(f"   Discovery method: {config['dashboard_selection']['method']}")

# Ensure directories exist
export_path = get_path('exported')
transformed_path = get_path('transformed')

print("\n📁 Ensuring directories exist...")
if ensure_directory_exists(export_path):
    print(f"   Created: {export_path}")
else:
    print(f"   Exists: {export_path}")

if ensure_directory_exists(transformed_path):
    print(f"   Created: {transformed_path}")
else:
    print(f"   Exists: {transformed_path}")

## Cell 3: Connect to Source Workspace & Discover Dashboards

In [ ]:
print("\n" + "="*80)
print("DISCOVERING DASHBOARDS")
print("="*80)

# Connect to source workspace (uses config)
print("\n🔗 Connecting to source workspace...")
source_client = create_workspace_client('source')
print(f"   ✅ Connected to: {config['source']['workspace_url']}\n")

# Discover dashboards (uses discovery method from config)
print(f"🔍 Discovering dashboards using method: {config['dashboard_selection']['method']}")

if config['dashboard_selection']['method'] == 'catalog_filter':
    catalog = config['dashboard_selection']['catalog_filter']['catalog']
    print(f"   Catalog filter: {catalog}")

dashboards = discover_dashboards(source_client)

print(f"\n✅ Found {len(dashboards)} dashboard(s)\n")

if dashboards:
    # Display summary
    df_summary = pd.DataFrame([
        {
            'Dashboard ID': d['id'],
            'Display Name': d['name'],
            'Path': d.get('path', 'N/A')
        }
        for d in dashboards
    ])
    display(df_summary)
else:
    print("⚠️  No dashboards found. Check your configuration.")

## Cell 4: Export Dashboards

In [ ]:
print("\n" + "="*80)
print("EXPORTING DASHBOARDS")
print("="*80)

if not dashboards:
    print("\n⚠️  No dashboards to export")
else:
    export_path = get_path('exported')
    export_results = []
    
    for i, dash in enumerate(dashboards, 1):
        dashboard_id = dash['id']
        print(f"\n[{i}/{len(dashboards)}] Exporting: {dash['name']}")
        print(f"   ID: {dashboard_id}")
        
        try:
            # Export dashboard JSON
            json_content, display_name, clean_name = export_dashboard(source_client, dashboard_id)
            
            # Save JSON to volume
            json_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}.lvdash.json"
            write_volume_file(json_file, json_content)
            print(f"   ✅ Exported JSON: {Path(json_file).name}")
            
            # Get and save permissions (if enabled in config)
            if config['permissions']['capture']:
                permissions = get_dashboard_permissions(source_client, dashboard_id)
                permissions['display_name'] = display_name
                
                perm_file = f"{export_path}/dashboard_{dashboard_id}_{clean_name}_permissions.json"
                write_volume_file(perm_file, json.dumps(permissions, indent=2))
                
                acl_count = len(permissions.get('access_control_list', []))
                print(f"   🔐 Permissions: {acl_count} ACL(s)")
            
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': display_name,
                'status': 'success',
                'json_file': json_file
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            export_results.append({
                'dashboard_id': dashboard_id,
                'name': dash['name'],
                'status': 'failed',
                'error': str(e)
            })
    
    # Summary
    successful = len([r for r in export_results if r['status'] == 'success'])
    failed = len([r for r in export_results if r['status'] == 'failed'])
    
    print("\n" + "="*80)
    print("EXPORT SUMMARY")
    print("="*80)
    print(f"\n✅ Successful: {successful}")
    print(f"❌ Failed: {failed}")
    print(f"\n📁 Exported files: {export_path}")

## Cell 5: Transform Dashboards

In [ ]:
print("\n" + "="*80)
print("TRANSFORMING DASHBOARDS")
print("="*80)

if not config['transformation']['enabled']:
    print("\n⚠️  Transformation disabled in config")
elif not export_results or successful == 0:
    print("\n⚠️  No dashboards to transform")
else:
    # Load mapping CSV
    mapping_csv_path = get_path('volume_base') + "/" + config['transformation']['mapping_csv']
    print(f"\n📋 Loading mapping CSV: {mapping_csv_path}")
    
    try:
        mappings = load_mapping_csv(mapping_csv_path)
        print(f"   ✅ Loaded {len(mappings)} mapping rule(s)\n")
        
        # Display sample mappings
        print("   Sample mappings:")
        for i, m in enumerate(mappings[:3], 1):
            old_ref = f"{m['old_catalog']}.{m['old_schema']}.{m.get('old_table', '*')}"
            new_ref = f"{m['new_catalog']}.{m['new_schema']}.{m.get('new_table', '*')}"
            print(f"      {i}. {old_ref} → {new_ref}")
        if len(mappings) > 3:
            print(f"      ... and {len(mappings) - 3} more\n")
        
    except Exception as e:
        print(f"   ❌ Failed to load mapping CSV: {e}")
        print(f"   💡 Ensure CSV file exists and follows format: old_catalog,old_schema,old_table,new_catalog,new_schema,new_table")
        raise
    
    # Transform each exported dashboard
    transformed_path = get_path('transformed')
    transform_results = []
    
    successful_exports = [r for r in export_results if r['status'] == 'success']
    
    for i, result in enumerate(successful_exports, 1):
        dashboard_id = result['dashboard_id']
        name = result['name']
        json_file = result['json_file']
        
        print(f"[{i}/{len(successful_exports)}] Transforming: {name}")
        
        try:
            # Read exported JSON
            json_content = read_volume_file(json_file)
            
            # Apply transformations
            transformed = transform_dashboard_json(json_content, mappings)
            
            # Save transformed version
            filename = Path(json_file).name
            transformed_file = f"{transformed_path}/{filename}"
            write_volume_file(transformed_file, transformed)
            
            # Check if content changed
            changes = "modified" if len(json_content) != len(transformed) else "no changes"
            print(f"   ✅ Transformed ({changes})")
            print(f"   💾 Saved: {Path(transformed_file).name}")
            
            transform_results.append({
                'dashboard_id': dashboard_id,
                'name': name,
                'status': 'success',
                'changes': changes
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            transform_results.append({
                'dashboard_id': dashboard_id,
                'name': name,
                'status': 'failed',
                'error': str(e)
            })
    
    # Summary
    successful_transforms = len([r for r in transform_results if r['status'] == 'success'])
    failed_transforms = len([r for r in transform_results if r['status'] == 'failed'])
    
    print("\n" + "="*80)
    print("TRANSFORM SUMMARY")
    print("="*80)
    print(f"\n✅ Successful: {successful_transforms}")
    print(f"❌ Failed: {failed_transforms}")
    print(f"\n📁 Transformed files: {transformed_path}")
    
    print("\n" + "="*80)
    print("NEXT STEPS - MANUAL IMPORT")
    print("="*80)
    print("\n1. Import transformed dashboards to target workspace:")
    print("   Option A: Use Bundle approach (recommended)")
    print("            → Run Bundle/Bundle_01_Export_and_Transform.ipynb")
    print("            → Run Bundle/Bundle_02_Generate_and_Deploy.ipynb")
    print("   Option B: Manual import via Databricks UI")
    print("            → Download files from volume")
    print("            → Import via Lakeview dashboard import feature")
    print("   Option C: Use SDK programmatically in your own script")
    print("\n2. After import, apply permissions:")
    print(f"   → Run notebooks/02_Apply_Permissions.ipynb")
    print(f"\n📋 Permissions available at: {export_path}/*_permissions.json")